# Tópicos Avançados em IA — Aula 05
## Processos gaussianos e otimização bayesiana

Prof. James Moraes de Almeida

Na Aula 04 a pergunta era *qual ponto rotular para o modelo ficar bom*. Hoje ela muda:

> **Onde está o máximo, se cada avaliação é cara?**

Roteiro:
1. Processo gaussiano: uma distribuição sobre funções.
2. Condicionar nos pontos medidos: média e incerteza.
3. Funções de aquisição: PI, EI e UCB.
4. O laço da otimização bayesiana, escrito à mão.
5. O mesmo com o Optuna.
6. Otimizando hiperparâmetros de um modelo.
7. **O enigma dos aços da Aula 04 — a resposta.**
8. Limites, para ir além — um **exercício com uma função caixa-preta** e as **apresentações de 19/11** (no fim).

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, Matern

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (11, 4)

# As cores da Aula 04, seguras para daltônicos: nada se distingue só pela cor — o que importa tem
# traço ou marcador próprio, ou o nome escrito ao lado.
AZUL, LARANJA, CINZA, TINTA = "#0072B2", "#E69F00", "#BBBBBB", "#1D5264"
LARANJA_ESCURO = "#9A5F00"                          # texto em laranja: o claro some no fundo branco
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=[TINTA, LARANJA, AZUL, "#7A7A7A", "#D55E00"])
VIRGULA = FuncFormatter(lambda v, _: f"{v:g}".replace(".", ",").replace("-", "−"))

def br(v, casas=2):
    return f"{v:.{casas}f}".replace(".", ",")

def virgula(*eixos):                                # vírgula decimal nos números dos dois eixos
    for ax in eixos:
        ax.xaxis.set_major_formatter(VIRGULA); ax.yaxis.set_major_formatter(VIRGULA)

## 1. Uma distribuição sobre funções

### Da normal multivariada ao processo gaussiano

Um vetor $\mathbf{f} = (f_1, \dots, f_n)$ tem distribuição **normal multivariada** $\mathcal{N}(\mathbf{m}, K)$
quando cada combinação linear dos seus componentes é normal. Ela é descrita por um vetor de médias
$\mathbf{m}$ e por uma **matriz de covariâncias** $K$, com $K_{ij} = \operatorname{Cov}(f_i, f_j)$.

Um **processo gaussiano** (GP, do inglês *Gaussian process*) leva a ideia para funções: $f \sim \mathcal{GP}(m, k)$ significa que, para
**qualquer** conjunto finito de pontos $x_1, \dots, x_n$,

$$\big(f(x_1), \dots, f(x_n)\big) \sim \mathcal{N}(\mathbf{m}, K), \qquad m_i = m(x_i), \quad K_{ij} = k(x_i, x_j).$$

A função média $m(x)$ costuma ser zero. Quem carrega a informação é o **kernel** $k(x, x')$: a
covariância entre os valores em dois pontos. Pontos próximos → covariância alta → valores parecidos.
Nem toda função serve de kernel: $K$ precisa ser positiva semidefinida para qualquer escolha de pontos.

### O kernel RBF

$$k_{\text{RBF}}(x, x') = \sigma^2 \exp\!\left(-\frac{(x - x')^2}{2\ell^2}\right).$$

* $\sigma^2$ (o `ConstantKernel`) é a variância: quanto a função se afasta da média.
* $\ell$ (o `length_scale`) é o **comprimento de correlação**: a distância em que a covariância cai
  para $e^{-1/2} \approx 0{,}61$ de $\sigma^2$: a uma distância $\ell$, a correlação entre $f(x)$ e $f(x')$ é 0,61;
  a $2\ell$, 0,14; a $3\ell$, 0,01.

**Sortear do prior** é sortear de $\mathcal{N}(0, K)$ numa grade fina: com a fatoração de Cholesky
$K = LL^\top$ e $\mathbf{z}$ normal padrão, $\mathbf{f} = L\mathbf{z}$ tem covariância $K$. É o que o
`sample_y` faz. Antes de medir qualquer coisa (o *prior*):

In [ ]:
xs = np.linspace(0, 10, 400)[:, None]
fig, eixos = plt.subplots(1, 3, figsize=(15, 3.9), sharey=True)
for ax, ell, frase in zip(eixos, [0.3, 1.0, 3.0], ["", "", ""]):
    gp = GaussianProcessRegressor(RBF(length_scale=ell), optimizer=None)
    amostras = gp.sample_y(xs, n_samples=5, random_state=0)
    ax.plot(xs, amostras, lw=1.6)
    ax.plot([0.4, 0.4 + ell], [-3.35, -3.35], color="k", lw=5, solid_capstyle="butt")   # a régua do ℓ
    ax.text(0.4 + ell + 0.15, -3.35, "ℓ", va="center", fontweight="bold", fontsize=14)
    ax.set_title(f"ℓ = {br(ell, 1)}")
    ax.set_xlabel("x"); virgula(ax)
eixos[0].set_ylabel("f(x)"); eixos[0].set_ylim(-3.7, 3.2)
plt.tight_layout()

**Dá para ver o ℓ nas curvas?** Contando: para o kernel RBF, uma curva sorteada cruza o zero, **em média**,
uma vez a cada $\pi\ell \approx 3\ell$ (a fórmula de Rice). É uma média: cada intervalo, sozinho, varia muito.
Conferimos contando os cruzamentos em 40 curvas longas para cada ℓ:

In [ ]:
xl = np.linspace(0, 60, 3000)[:, None]
for ell in [0.3, 1.0, 3.0]:
    curvas_l = GaussianProcessRegressor(RBF(length_scale=ell), optimizer=None).sample_y(xl, n_samples=40, random_state=0)
    por_unidade = (np.diff(np.sign(curvas_l), axis=0) != 0).sum(axis=0).mean() / 60
    print(f"ℓ = {br(ell, 1)}: um cruzamento do zero a cada {br(1 / por_unidade)} (= {br(1 / por_unidade / ell)} ℓ; "
          f"a fórmula dá π ≈ 3,14 ℓ)")
    if ell == 1.0:                                     # cada intervalo, sozinho, varia muito
        inter = np.concatenate([np.diff(xl[np.flatnonzero(np.diff(np.sign(c)) != 0), 0]) for c in curvas_l.T])
        print(f"   com ℓ = 1, intervalos entre zeros: mediana {br(np.median(inter))}; 10% abaixo de "
              f"{br(np.percentile(inter, 10))} e 10% acima de {br(np.percentile(inter, 90))}")

### Dois pontos: o quanto andam juntos

O kernel é a covariância entre os valores da função em dois pontos: $\operatorname{Cov}(f(x), f(x')) = k(x, x')$.
Com $\sigma = 1$, é também a **correlação**. Conferimos: sorteamos muitos pares $(f(x), f(x'))$ da normal
conjunta, com $x'$ perto e longe de $x$ ($\ell = 1$), e medimos a correlação dos sorteios.

In [ ]:
g = np.random.default_rng(0)
for d in [0.4, 1.0, 4.0]:
    k_d = np.exp(-d ** 2 / 2)                          # RBF com σ = 1 e ℓ = 1
    pares = g.multivariate_normal([0, 0], [[1, k_d], [k_d, 1]], size=20_000)
    print(f"x′ − x = {br(d, 1)}: correlação nos sorteios {br(np.corrcoef(pares.T)[0, 1], 3)} · kernel {br(k_d, 3)}")

**O que isso muda quando se mede.** Queremos palpite e dúvida em **todo** $x$, mas cada medida custa (um
experimento, uma simulação): só se mede em poucos pontos. Então $x$ é um ponto qualquer **ainda sem medida**, e a
pergunta é o quanto uma medida feita em outro ponto, $x'$, diminui a dúvida em $x$. Medir no próprio $x$
($x' - x = 0$) zera a dúvida ali, e só ali. Para dois pontos de uma normal conjunta com $\sigma = 1$, depois de
conhecer $f(x')$ o desvio em $x$ vira $\sqrt{1 - k^2}$. Conferimos com o próprio GP, ajustado a uma medida só:

In [ ]:
for d in [0.0, 0.4, 1.0, 2.0, 4.0]:
    gp1 = GaussianProcessRegressor(RBF(length_scale=1.0), alpha=1e-10, optimizer=None).fit([[0.0]], [0.0])
    _, sd_x = gp1.predict([[d]], return_std=True)
    k_d = np.exp(-d ** 2 / 2)
    print(f"x′ − x = {br(d, 1)} (= {br(d, 1)} ℓ): a dúvida em x vai de 1 para {br(sd_x[0])} "
          f"(fórmula √(1 − k²) = {br(np.sqrt(1 - k_d ** 2))})")

O `ℓ` pequeno gera funções nervosas; o grande, funções suaves.

### O kernel Matern

O kernel **Matern** com $\nu = 5/2$ é o padrão em otimização bayesiana. Com $r = |x - x'|$:

$$k_{5/2}(r) = \sigma^2\left(1 + \frac{\sqrt{5}\,r}{\ell} + \frac{5r^2}{3\ell^2}\right)\exp\!\left(-\frac{\sqrt{5}\,r}{\ell}\right).$$

O parâmetro $\nu$ controla a suavidade: as funções sorteadas têm derivadas até a ordem $\lceil\nu\rceil - 1$.
Com $\nu = 5/2$, duas derivadas; o RBF é o limite $\nu \to \infty$, com **infinitas** derivadas — suave
demais para a maioria das medidas reais. No código, `Matern(nu=2.5)`.

### Ajustar $\ell$ e $\sigma^2$: a verossimilhança marginal

Na prática não escolhemos $\ell$ à mão: o GP escolhe o $\ell$ que maximiza a **verossimilhança
marginal** dos dados medidos $\mathbf{y}$,

$$\log p(\mathbf{y} \mid X) = \underbrace{-\tfrac{1}{2}\,\mathbf{y}^\top K^{-1}\mathbf{y}}_{\text{ajuste aos dados}}
\;\underbrace{-\;\tfrac{1}{2}\log\lvert K\rvert}_{\text{penalidade de complexidade}} \;-\; \tfrac{n}{2}\log 2\pi .$$

O primeiro termo premia kernels que explicam os dados; o segundo pune kernels flexíveis demais. É
uma navalha de Occam embutida — e, com poucos pontos, ela pode não ter informação para decidir
(seção 2).

## 2. Condicionar nos pontos medidos

Medimos a função em alguns pontos. O GP passa a descrever só as funções **compatíveis**
com essas medidas: o *posterior*. Em cada `x` ele devolve uma média `μ(x)` (o palpite) e
um desvio padrão `σ(x)` (quanto não sabemos).

As duas vêm da regra de condicionamento da normal multivariada. Com as medidas $\mathbf{y}$ nos pontos
$X$, $K = k(X, X)$, $\mathbf{k}_* = k(X, x_*)$ e um ruído de variância $\sigma_n^2$:

$$\mu(x_*) = \mathbf{k}_*^\top (K + \sigma_n^2 I)^{-1}\mathbf{y}, \qquad
\sigma^2(x_*) = k(x_*, x_*) - \mathbf{k}_*^\top (K + \sigma_n^2 I)^{-1}\mathbf{k}_* .$$

* A média é uma **combinação ponderada das medidas**, com pesos maiores para as mais próximas.
* A variância é a do prior **menos** o que as medidas explicam. Sobre uma medida sem ruído,
  $\sigma^2 = 0$ — por isso a faixa se fecha lá.
* No código, $\sigma_n^2$ é o `alpha=1e-6`: quase zero (a função não tem ruído), mas não zero, para a
  matriz não ficar numericamente singular.

A função escondida desta aula:

In [ ]:
def f(x):
    """A função 'cara'. Fingimos que cada chamada custa uma semana."""
    return -(np.sin(x) + np.sin(10 * x / 3))

A, B = 2.7, 7.5
grade = np.linspace(A, B, 1000)

def ajustar_gp(X, y):
    """Amplitude fixa em σ² = 1 (a função varia entre −1 e 2) e ℓ ajustado, mas preso
    em [0,2; 2]: com três ou quatro medidas não há informação para ajustar os dois,
    e o ajuste livre degenera (a faixa some ou vira um bloco)."""
    kernel = ConstantKernel(1.0, "fixed") * Matern(length_scale=1.0, length_scale_bounds=(0.2, 2.0), nu=2.5)
    return GaussianProcessRegressor(kernel, alpha=1e-6, normalize_y=False,
                                    n_restarts_optimizer=5, random_state=0).fit(X[:, None], y)

def mostrar_gp(X, ax=None, titulo=""):
    ax = ax or plt.gca()
    gp = ajustar_gp(X, f(X))
    mu, sd = gp.predict(grade[:, None], return_std=True)
    ax.plot(grade, f(grade), "k--", lw=1, label="f (escondida)")
    ax.plot(grade, mu, color=TINTA, lw=2, label="média μ")
    ax.fill_between(grade, mu - 2 * sd, mu + 2 * sd, color=AZUL, alpha=0.18, lw=0, label="μ ± 2σ (~95% das funções)")
    ax.plot(X, f(X), "o", color=LARANJA, mec="#1A1A1A", mew=1, ms=9, label="medidas", zorder=3)
    ax.set_title(titulo); virgula(ax); return gp, mu, sd

X3 = np.array([3.2, 4.2, 6.6])
gp3 = ajustar_gp(X3, f(X3))
curvas = gp3.sample_y(grade[:, None], n_samples=30, random_state=1)
fig, eixos = plt.subplots(1, 3, figsize=(16, 4.2), sharey=True, gridspec_kw=dict(width_ratios=[1, 1, 0.85]))
eixos[0].plot(grade, curvas, color=TINTA, alpha=0.35, lw=1)
eixos[0].plot(grade, f(grade), "k--", lw=1)
eixos[0].plot(X3, f(X3), "o", color=LARANJA, mec="#1A1A1A", mew=1, ms=9, zorder=3)
eixos[0].set_title("30 funções sorteadas que passam pelas medidas")
mostrar_gp(X3, eixos[1], "a média μ e a faixa μ ± 2σ delas")
mostrar_gp(np.array([3.2, 4.2, 4.8, 5.6, 6.2, 6.6]), eixos[2], "6 medidas: a faixa estreita")
eixos[1].legend(loc="lower left", fontsize=10)
for ax in eixos:
    ax.set_xlabel("x")
eixos[0].set_ylabel("f(x)"); virgula(eixos[0])
plt.tight_layout();

**Repare:** a faixa se fecha sobre cada medida (lá não há dúvida) e se abre longe delas.
É essa incerteza que a otimização bayesiana usa para decidir onde medir.

**De onde vêm μ e σ.** Medir é ficar só com as funções do prior que passam pelas medidas. Em cada
x, a média dessas funções é μ(x) e o quanto elas se espalham é σ(x). Antes de medir, σ(x) = σ em todo
lugar; depois, encolhe perto das medidas, onde as funções que sobraram concordam. Conferimos com
2.000 funções sorteadas do posterior:

In [ ]:
grade_c = grade[::10, None]
sorteios = gp3.sample_y(grade_c, n_samples=2000, random_state=2)
mu_c, sd_c = gp3.predict(grade_c, return_std=True)
print(f"maior diferença entre a média das 2.000 funções e μ: {br(np.abs(sorteios.mean(1) - mu_c).max(), 3)}")
print(f"maior diferença entre o espalhamento delas e σ:      {br(np.abs(sorteios.std(1) - sd_c).max(), 3)}")

### As fórmulas, à mão

Refazemos o posterior de 3 medidas com as fórmulas acima e comparamos com o `scikit-learn`:

In [ ]:
X3 = np.array([3.2, 4.2, 6.6]); y3 = f(X3)
gp3 = ajustar_gp(X3, y3)
k = gp3.kernel_                                     # o kernel com o ℓ já ajustado
K = k(X3[:, None]) + 1e-6 * np.eye(3)               # K + σ²_n I
Ks = k(grade[:, None], X3[:, None])                 # k(x*, X) em toda a grade
mu_formula = Ks @ np.linalg.solve(K, y3)
var_formula = np.diag(k(grade[:, None])) - np.einsum("ij,ji->i", Ks, np.linalg.solve(K, Ks.T))
mu_sk, sd_sk = gp3.predict(grade[:, None], return_std=True)
print(f"kernel ajustado: {k}")
print(f"média:  maior diferença fórmula × sklearn {np.abs(mu_formula - mu_sk).max():.1e}")
print(f"desvio: maior diferença fórmula × sklearn {np.abs(np.sqrt(np.maximum(var_formula, 0)) - sd_sk).max():.1e}")
lml = -0.5 * y3 @ np.linalg.solve(K, y3) - 0.5 * np.linalg.slogdet(K)[1] - 1.5 * np.log(2 * np.pi)
print(f"log da verossimilhança marginal: fórmula {lml:.4f} · sklearn {gp3.log_marginal_likelihood_value_:.4f}")

### Por que o $\ell$ precisa de limites com poucas medidas

O `ajustar_gp` prende $\ell$ entre 0,2 e 2. Veja a verossimilhança marginal das 3 medidas para cada
$\ell$ fixo:

In [ ]:
for ell in [0.05, 0.1, 0.2, 0.4, 0.8, 1.6, 3.2]:
    k_fixo = ConstantKernel(1.0, "fixed") * Matern(length_scale=ell, length_scale_bounds="fixed", nu=2.5)
    v = GaussianProcessRegressor(k_fixo, alpha=1e-6).fit(X3[:, None], y3).log_marginal_likelihood_value_
    print(f"ℓ = {ell:4.2f}   log p(y | X) = {v:7.3f}")

Abaixo de $\ell \approx 0{,}4$ a verossimilhança é **praticamente plana**: com três pontos a
distâncias de 1 a 3,4, qualquer $\ell$ pequeno trata as medidas como independentes e as explica
igualmente bem. Os dados não têm informação para escolher $\ell$, e o otimizador vai para a borda do
intervalo permitido. Sem limite, ele iria para $\ell \to 0$ — e a faixa viraria um bloco, com a média
em zero longe de cada medida. Com mais medidas a verossimilhança ganha forma e o problema some.

## 3. Funções de aquisição

Com `μ(x)` e `σ(x)` na mão, onde medir? Seja `f*` o melhor valor medido até agora.

| aquisição | fórmula | o que premia |
|---|---|---|
| **PI** — probabilidade de melhora | `Φ(z)`, com `z = (μ − f* − ξ)/σ` | chance de passar do recorde, por pouco que seja |
| **EI** — melhora esperada | `(μ − f* − ξ)Φ(z) + σ φ(z)` | chance **e** tamanho da melhora |
| **UCB** — limite superior | `μ + κσ` | otimismo: "pode ser tão bom quanto μ + κσ" |

`Φ` e `φ` são a acumulada e a densidade da normal. O `κ` do UCB é o **dial explorar ×
explotar**: `κ = 0` só olha a média (explota); `κ` grande só olha a incerteza (explora).

### De onde vem a fórmula do EI

Em cada $x$, o GP diz que $f(x) \sim \mathcal{N}(\mu, \sigma^2)$. A **melhora** sobre o recorde é
$I = \max(f(x) - f^* - \xi,\ 0)$ — zero se não passar do recorde. A melhora **esperada** é a integral
dessa melhora sob a normal:

$$\text{EI}(x) = \int_{f^*+\xi}^{\infty} (f - f^* - \xi)\,\mathcal{N}(f;\mu,\sigma^2)\,df
= (\mu - f^* - \xi)\,\Phi(z) + \sigma\,\varphi(z), \qquad z = \frac{\mu - f^* - \xi}{\sigma}.$$

Os dois termos são o "explotar" (média alta) e o "explorar" (incerteza alta). Num ponto já medido,
$\sigma = 0$ e $\mu = f(x) \le f^*$: o EI é zero — não há o que ganhar medindo de novo. O $\xi$ (o `xi`)
exige uma melhora mínima e empurra um pouco para a exploração. A PI é só a probabilidade
$P(f > f^* + \xi) = \Phi(z)$, sem o tamanho da melhora.

In [ ]:
def aquisicao(mu, sd, melhor, tipo="EI", xi=0.01, kappa=2.0):
    sd = np.maximum(sd, 1e-12)
    z = (mu - melhor - xi) / sd
    if tipo == "PI":
        return norm.cdf(z)
    if tipo == "EI":
        return (mu - melhor - xi) * norm.cdf(z) + sd * norm.pdf(z)
    if tipo == "UCB":
        return mu + kappa * sd
    raise ValueError(tipo)

X = np.array([3.2, 4.2, 6.6])
fig = plt.figure(figsize=(15, 6.4))
gs = fig.add_gridspec(2, 3, height_ratios=[1.0, 1], hspace=0.75, wspace=0.22)
ax0 = fig.add_subplot(gs[0, :])
gp, mu, sd = mostrar_gp(X, ax0, "posterior com 3 medidas")
ax0.legend(loc="upper right", fontsize=9, ncol=4)
FORMULA = {"PI": r"PI $= \Phi(z)$",
           "EI": r"EI $= (\mu - f^* - \xi)\,\Phi(z) + \sigma\,\varphi(z)$",
           "UCB": r"UCB $= \mu + \kappa\,\sigma$"}
for j, tipo in enumerate(["PI", "EI", "UCB"]):
    ax = fig.add_subplot(gs[1, j], sharex=ax0)
    a = aquisicao(mu, sd, f(X).max(), tipo)
    ax.plot(grade, a, color=TINTA, lw=2); ax.axvline(grade[a.argmax()], color=LARANJA_ESCURO, ls="--", lw=1.8)
    ax.set_title(f"{FORMULA[tipo]}\npróximo ponto: x = {br(grade[a.argmax()])}", fontsize=16)
    virgula(ax)

### Uma de cada vez

As figuras abaixo separam as três regras (são as dos slides). Todas usam o posterior de 3 medidas
acima e o mesmo `f*`, o melhor valor medido até agora.

In [ ]:
f_star = f(X).max()
dist_medida = np.abs(grade[:, None] - X[None, :]).min(axis=1)
i_mu = np.argmax(np.where(sd > 0.4, mu, -np.inf))                  # explotar: o palpite mais alto onde ainda há dúvida
                                                                    # (em cima da medida, σ = 0: medir de novo não ensina nada)
i_sd = np.argmax(dist_medida * (sd >= sd.max() - 1e-3))             # explorar: a faixa mais larga, longe das medidas
fig, ax = plt.subplots(figsize=(13, 4))
mostrar_gp(X, ax, "")
ax.axhline(f_star, color=LARANJA_ESCURO, ls=":", lw=2)
ax.text(7.5, f_star + 0.12, "f*: o melhor medido até agora", ha="right", color=LARANJA_ESCURO, fontweight="bold")
ax.plot(grade[i_mu], mu[i_mu], "^", ms=16, color=TINTA, mec="white", mew=1.5, zorder=5)
ax.annotate("▲ perto do recorde, palpite alto:\nexplotar", (grade[i_mu], mu[i_mu]), xytext=(45, -95),
            textcoords="offset points", fontweight="bold", color=TINTA, arrowprops=dict(arrowstyle="->", color=TINTA))
ax.plot(grade[i_sd], mu[i_sd], "s", ms=14, color="white", mec=TINTA, mew=2.5, zorder=5)
ax.annotate("□ a faixa mais larga (σ alto):\nexplorar", (grade[i_sd], mu[i_sd]), xytext=(-60, -75),
            textcoords="offset points", fontweight="bold", color=TINTA, arrowprops=dict(arrowstyle="->", color=TINTA))
ax.set_xlabel("x"); ax.set_ylabel("f(x)")
print(f"f* = {br(f_star)} · palpite mais alto em x = {br(grade[i_mu])} (μ = {br(mu[i_mu])}) · "
      f"faixa mais larga em x = {br(grade[i_sd])} (σ = {br(sd[i_sd])})")
print(f"se medirmos no ▲: f = {br(f(grade[i_mu]))} · se medirmos no ■: f = {br(f(grade[i_sd]))} · "
      f"o máximo: {br(f(grade).max())} em x = {br(grade[f(grade).argmax()])}")

In [ ]:
ucb = mu + 2 * sd
fig, ax = plt.subplots(figsize=(13, 4))
mostrar_gp(X, ax, "")
ax.plot(grade, ucb, color=TINTA, lw=3.2)
ax.text(7.5, ucb[-1] + 0.15, "UCB = μ + 2σ: o topo da faixa", ha="right", color=TINTA, fontweight="bold")
i_u = ucb.argmax()
ax.axvline(grade[i_u], color=LARANJA_ESCURO, ls="--", lw=1.8)
ax.plot(grade[i_u], ucb[i_u], "v", ms=14, color=LARANJA, mec="#1A1A1A", zorder=5)
ax.text(grade[i_u] + 0.06, -1.9, f"mede aqui: x = {br(grade[i_u])}", color=LARANJA_ESCURO, fontweight="bold")
ax.set_xlabel("x"); ax.set_ylabel("f(x)"); ax.set_ylim(-2.3, 3.0)
print(f"κ = 2: o UCB é máximo em x = {br(grade[i_u])} · maior diferença entre μ + 2σ e a borda da faixa: "
      f"{np.abs(ucb - aquisicao(mu, sd, f_star, 'UCB', kappa=2)).max():.1e}")

In [ ]:
x0 = 5.0                                              # um ponto no meio do vão, para desenhar a normal
i0 = np.abs(grade - x0).argmin()
m0, s0 = float(np.round(mu[i0], 2)) + 0.0, float(sd[i0])   # + 0,0: sem "−0,00"
yy = np.linspace(m0 - 3.2 * s0, m0 + 3.2 * s0, 400)
dens = norm.pdf(yy, m0, s0); larg = 0.9 / dens.max()   # a normal deitada, com 0,9 de largura no eixo x

def normal_deitada(ax, sombra):
    ax.plot(x0 + larg * dens, yy, color=TINTA, lw=2)
    ax.plot([x0, x0], [yy[0], yy[-1]], color=TINTA, lw=1)
    acima = yy >= f_star
    if sombra == "pi":
        ax.fill_betweenx(yy[acima], x0, x0 + larg * dens[acima], color=LARANJA, alpha=0.6, lw=0)
    else:                                             # EI: a sombra mais forte quanto mais passa de f*
        ys = yy[acima]
        for a, b in zip(ys[:-1], ys[1:]):
            ax.fill_betweenx([a, b], x0, x0 + larg * norm.pdf(a, m0, s0),
                             color=LARANJA, alpha=min(0.95, 0.15 + 0.8 * (a - f_star) / (ys[-1] - f_star)), lw=0)
    ax.axhline(f_star, color=LARANJA_ESCURO, ls=":", lw=2)
    ax.text(2.75, f_star + 0.1, "f*", color=LARANJA_ESCURO, fontweight="bold")

pi0 = norm.cdf((m0 - f_star) / s0)
fig, (a0, a1) = plt.subplots(1, 2, figsize=(15, 4.2), gridspec_kw=dict(width_ratios=[1.25, 1]))
mostrar_gp(X, a0, f"em x₀ = {br(x0, 1)}, o valor é sorteado de uma normal (μ = {br(m0)}, σ = {br(s0)})")
normal_deitada(a0, "pi")
a0.text(x0 + 1.0, f_star + 0.6, f"área sombreada = PI(x₀) = {br(pi0)}:\na chance de passar de f*",
        color=LARANJA_ESCURO, fontweight="bold")
a0.set_xlabel("x"); a0.set_ylabel("f(x)")
pi = aquisicao(mu, sd, f_star, "PI")              # com o ξ = 0,01 do laço: sem ele, a PI escolhe o próprio recorde
a1.plot(grade, pi, color=TINTA, lw=2); a1.axvline(grade[pi.argmax()], color=LARANJA_ESCURO, ls="--", lw=1.8)
a1.plot(x0, pi0, "o", color=LARANJA, mec="#1A1A1A", ms=9)
a1.set_title(f"PI em todo x: máxima em x = {br(grade[pi.argmax()])}, colada no recorde"); a1.set_xlabel("x"); virgula(a1)
plt.tight_layout()
mc = (np.random.default_rng(3).normal(m0, s0, 1_000_000) > f_star).mean()
print(f"x₀ = {br(x0, 1)}: μ = {br(m0)}, σ = {br(s0)}, f* = {br(f_star)} · PI pela fórmula Φ((μ − f*)/σ) = {br(pi0, 4)} · "
      f"por sorteio = {br(mc, 4)} · PI máxima em x = {br(grade[pi.argmax()])}")

In [ ]:
z0 = (m0 - f_star) / s0
ei0 = (m0 - f_star) * norm.cdf(z0) + s0 * norm.pdf(z0)
fig, (a0, a1) = plt.subplots(1, 2, figsize=(15, 4.2), gridspec_kw=dict(width_ratios=[1.25, 1]))
mostrar_gp(X, a0, f"em x₀ = {br(x0, 1)}: quanto se espera ganhar além de f*?")
normal_deitada(a0, "ei")
a0.text(x0 + 1.0, f_star + 0.6, f"EI(x₀) = {br(ei0, 3)}: cada melhora\nvezes a sua chance, somadas",
        color=LARANJA_ESCURO, fontweight="bold")
a0.set_xlabel("x"); a0.set_ylabel("f(x)")
ei = aquisicao(mu, sd, f_star, "EI")
a1.plot(grade, ei, color=TINTA, lw=2); a1.axvline(grade[ei.argmax()], color=LARANJA_ESCURO, ls="--", lw=1.8)
a1.plot(x0, ei0, "o", color=LARANJA, mec="#1A1A1A", ms=9)
a1.set_title(f"EI em todo x: máxima em x = {br(grade[ei.argmax()])}"); a1.set_xlabel("x"); virgula(a1)
plt.tight_layout()
mc = np.maximum(np.random.default_rng(4).normal(m0, s0, 1_000_000) - f_star, 0).mean()
print(f"x₀ = {br(x0, 1)}: z = {br(z0)} · EI pela fórmula = {br(ei0, 4)} · por sorteio = {br(mc, 4)} · "
      f"EI máxima em x = {br(grade[ei.argmax()])}")

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
mostrar_gp(X, ax, "")
ax.axhline(f_star, color=LARANJA_ESCURO, ls=":", lw=1.5)
escolhas = [("PI", "s", ":"), ("EI", "D", "--"), ("UCB", "v", "-.")]
for k, (tipo, marca, traco) in enumerate(escolhas):
    xa = grade[aquisicao(mu, sd, f_star, tipo).argmax()]
    ax.axvline(xa, color="#555555", ls=traco, lw=1.6)
    ax.plot(xa, 2.35 - 0.35 * k, marca, ms=11, color=LARANJA, mec="#1A1A1A")
    ax.text(xa + 0.07, 2.35 - 0.35 * k, f"{tipo}: x = {br(xa)}", va="center", fontweight="bold")
ax.set_ylim(-2.3, 2.6); ax.set_xlabel("x"); ax.set_ylabel("f(x)")

Conferimos a fórmula contra um sorteio de um milhão de valores da normal, no ponto que o EI escolheu:

In [ ]:
ei = aquisicao(mu, sd, f(X).max(), "EI")
i = ei.argmax()
sorteio = np.random.default_rng(0).normal(mu[i], sd[i], 1_000_000)
ei_mc = np.maximum(sorteio - f(X).max() - 0.01, 0).mean()
print(f"x = {grade[i]:.3f}: μ = {mu[i]:.3f}, σ = {sd[i]:.3f}, f* = {f(X).max():.3f}")
print(f"EI pela fórmula {ei[i]:.4f} · EI por Monte Carlo {ei_mc:.4f}")

## 4. O laço, escrito à mão

```
medidas iniciais → ajusta o GP → maximiza a aquisição → mede lá → repete
```

In [ ]:
def otimizacao_bayesiana(X0, n_iter=10, tipo="EI", **kw):
    X = list(X0)
    for _ in range(n_iter):
        gp = ajustar_gp(np.array(X), f(np.array(X)))
        mu, sd = gp.predict(grade[:, None], return_std=True)
        X.append(grade[aquisicao(mu, sd, f(np.array(X)).max(), tipo, **kw).argmax()])
    return np.array(X)

def intervalos(ks):
    ks, partes, i = sorted(ks), [], 0
    while i < len(ks):
        j = i
        while j + 1 < len(ks) and ks[j + 1] == ks[j] + 1:
            j += 1
        partes.append(str(ks[i]) if j == i else f"{ks[i]}–{ks[j]}")
        i = j + 1
    return ", ".join(partes)

def rotular(ax, novos):
    """Numera as medidas novas na ordem; as vizinhas ganham um rótulo só ("4, 6–10")."""
    grupos = []
    for k, x in sorted(enumerate(novos, 1), key=lambda par: par[1]):
        if grupos and x - grupos[-1][-1][1] < 0.15:
            grupos[-1].append((k, x))
        else:
            grupos.append([(k, x)])
    for g in grupos:
        x = max(g, key=lambda par: f(par[1]))[1]
        ax.annotate(intervalos([k for k, _ in g]), (x, f(x)), textcoords="offset points", xytext=(0, 9),
                    ha="center", fontweight="bold")

X = otimizacao_bayesiana([3.2, 4.2, 6.6], tipo="UCB")
fig, eixos = plt.subplots(1, 2, figsize=(15, 4))
mostrar_gp(X, eixos[0], "depois de 10 iterações (UCB, κ = 2)")
rotular(eixos[0], X[3:])
eixos[0].set_ylim(top=2.5)
eixos[1].plot(np.maximum.accumulate(f(X)), "o-", color=TINTA, lw=2)
eixos[1].axhline(f(grade).max(), color="k", ls="--", label="máximo verdadeiro"); virgula(eixos[1])
eixos[1].set_xlabel("número de medidas"); eixos[1].set_ylabel("melhor valor até agora"); eixos[1].legend();

Uma repetição só é uma anedota. Com 20 repetições, cada uma com outros 3 pontos iniciais ao acaso,
quantas vezes cada aquisição acha o máximo global em até 10 iterações?

In [ ]:
F_MAX = f(grade).max()
rng = np.random.default_rng(0)
partidas = [np.sort(rng.uniform(A, B, 3)) for _ in range(20)]
for tipo in ["EI", "UCB"]:
    achou = [f(otimizacao_bayesiana(x0, 10, tipo)).max() > F_MAX - 1e-3 for x0 in partidas]
    print(f"{tipo}: achou o máximo em {sum(achou)} de 20 repetições")

**Pergunta:** o EI fica preso em algumas repetições. Onde? Plote uma delas. Por que o UCB,
com `κ = 2`, escapa mais vezes?

### O dial `κ`

Se explorar é bom, explorar mais é melhor? As mesmas 20 repetições, 10 iterações cada:

In [ ]:
achou_kappa = {}
for kappa in [0.5, 1, 2, 4]:
    achou_kappa[kappa] = sum(f(otimizacao_bayesiana(x0, 10, "UCB", kappa=kappa)).max() > F_MAX - 1e-3
                             for x0 in partidas)
    print(f"UCB com κ = {kappa}: achou o máximo em {achou_kappa[kappa]} de 20 repetições")

In [ ]:
# em cima: o mesmo posterior de 3 medidas, com o topo μ + κσ para dois κ
X3k = np.array([3.2, 4.2, 6.6])
gpk = ajustar_gp(X3k, f(X3k))
mu_k, sd_k = gpk.predict(grade[:, None], return_std=True)
# embaixo: a primeira das 20 repetições em que κ = 0,5 fica preso, κ = 2 acha o máximo e κ = 4 não
achou = lambda X: f(X).max() > F_MAX - 1e-3
rodadas_k = {}
for r, x0 in enumerate(partidas):
    rodadas_k = {k: otimizacao_bayesiana(x0, 10, "UCB", kappa=k) for k in (0.5, 2, 4)}
    if not achou(rodadas_k[0.5]) and achou(rodadas_k[2]) and not achou(rodadas_k[4]):
        break
print(f"repetição escolhida: {r + 1} de 20 (pontos iniciais x = {', '.join(br(v) for v in x0)})")

fig = plt.figure(figsize=(16, 7.6))
gs = fig.add_gridspec(2, 3, height_ratios=[1, 1], hspace=0.45, wspace=0.18)
a = fig.add_subplot(gs[0, :])
a.fill_between(grade, mu_k - 2 * sd_k, mu_k + 2 * sd_k, color=AZUL, alpha=0.12, lw=0)
a.plot(grade, mu_k, color=TINTA, lw=2)
a.plot(X3k, f(X3k), "o", color=LARANJA, mec="#1A1A1A", ms=9, zorder=4)
for k, traco, y_txt in [(0.5, "--", 1.05), (4, "-", 4.55)]:
    topo = mu_k + k * sd_k
    a.plot(grade, topo, color="#555555", ls=traco, lw=2.2)
    i = topo.argmax()
    a.plot(grade[i], topo[i], "v", ms=13, color=LARANJA, mec="#1A1A1A", zorder=5)
    nk = br(k, 1) if k % 1 else str(int(k))
    a.text(4.75, y_txt, f"κ = {nk}: o topo μ + {nk}σ ({'tracejado' if traco == '--' else 'contínuo'}) — mede em x = {br(grade[i])}",
           fontweight="bold", va="center")
a.set_ylim(-2.3, 5.3)
a.set_title("UCB = μ + κσ: κ multiplica a incerteza", pad=10); a.set_ylabel("f(x)"); virgula(a)
for j, (k, nome) in enumerate([(0.5, "κ = 0,5: amontoa no 1º morro"), (2, "κ = 2: acha o pico"),
                               (4, "κ = 4: espalha e não refina")]):
    ax = fig.add_subplot(gs[1, j])
    Xk = rodadas_k[k]
    ax.plot(grade, f(grade), "k--", lw=1)
    ax.plot(Xk[:3], f(Xk[:3]), "o", color="white", mec="#1A1A1A", ms=8)
    ax.plot(Xk[3:], f(Xk[3:]), "o", color=LARANJA, mec="#1A1A1A", ms=9, zorder=3)
    rotular(ax, Xk[3:])
    valores, vezes = np.unique(np.round(Xk[3:], 6), return_counts=True)
    if vezes.max() > 1:                               # o laço mediu o mesmo x várias vezes
        ax.text(0.03, 0.05, f"{vezes.max()} das 10 medidas no mesmo x = {br(valores[vezes.argmax()])}",
                transform=ax.transAxes, fontweight="bold", color=LARANJA_ESCURO)
        print(f"κ = {br(k, 1) if k % 1 else int(k)}: {vezes.max()} das 10 medidas no mesmo x = {br(valores[vezes.argmax()], 3)}")
    ax.set_title(nome); ax.set_xlabel("x"); ax.set_ylim(-2.2, 2.6); virgula(ax)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))
ks = list(achou_kappa)
ax.bar(range(len(ks)), [achou_kappa[k] for k in ks], color=TINTA, width=0.6)
for i, k in enumerate(ks):
    ax.text(i, achou_kappa[k] + 0.4, f"{achou_kappa[k]} de 20", ha="center", fontweight="bold")
ax.axhline(20, color="k", ls=":", lw=1.2)
rotulo = {0.5: "\nexplota cedo demais", 1: "", 2: "\nequilíbrio", 4: "\nexplora demais"}
ax.set_xticks(range(len(ks)), [f"κ = {br(k, 1) if k % 1 else int(k)}{rotulo[k]}" for k in ks])
ax.set_ylim(0, 22.5); ax.set_ylabel("acharam o máximo")
ax.set_title("UCB, 10 iterações: em quantas das 20 repetições achou o máximo");

**Nem pouco, nem muito.** Com `κ` pequeno o laço se agarra ao primeiro morro bom; com
`κ = 4` ele gasta as 10 medidas mapeando regiões sem chance, e não chega a refinar o pico.
O melhor `κ` depende do orçamento: com 100 medidas, explorar mais custaria menos.

**E a teoria?** Srinivas e colegas (2010) provaram que o UCB converge para o máximo se $\kappa$
**cresce devagar** com o número de iterações $t$ (como $\sqrt{\log t}$): explorar cada vez mais,
mas nunca parar de explorar. É uma garantia **assintótica**, para orçamento que cresce sem fim.
O resultado acima é outra pergunta — **orçamento fixo de 10** — e nela um $\kappa$ fixo moderado
ganha. As duas coisas são verdade ao mesmo tempo.

Um detalhe do nosso laço: ele maximiza a aquisição numa grade de 1000 pontos. Em mais dimensões
não há grade que caiba; as bibliotecas usam otimização numérica com vários pontos de partida.

## 5. O mesmo com o Optuna

Na prática ninguém escreve o laço: o Optuna faz isso para qualquer função. O `GPSampler`
é otimização bayesiana com GP; o `TPESampler` (o padrão) é outra família bayesiana, mais
barata; o `RandomSampler` é a referência.

**Como o TPE funciona** (Bergstra e colegas, 2011). Em vez de modelar $f(x)$, ele separa as
tentativas já feitas em "boas" (a melhor fração, uns 10–25%) e "ruins", e estima duas densidades de
probabilidade sobre $x$: $\ell(x)$, onde as boas estão, e $g(x)$, onde as ruins estão. O próximo ponto
maximiza $\ell(x)/g(x)$ — "parecido com as boas, diferente das ruins". Os autores mostram que isso
equivale a maximizar a melhora esperada sob o modelo deles. Custa pouco por iteração e lida bem com
variáveis discretas e condicionais — por isso é o padrão.

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objetivo(trial):
    return f(trial.suggest_float("x", A, B))

# O GPSampler e o TPESampler sorteiam ao acaso as primeiras n_startup_trials tentativas (o padrão é 10)
# e só depois usam o modelo; com 3, ficam como o laço à mão: 3 iniciais + 10 escolhidas.
for nome, amostrador in [("aleatório", optuna.samplers.RandomSampler(seed=0)),
                         ("TPE", optuna.samplers.TPESampler(seed=0, n_startup_trials=3)),
                         ("GP", optuna.samplers.GPSampler(seed=0, n_startup_trials=3))]:
    estudo = optuna.create_study(direction="maximize", sampler=amostrador)
    estudo.optimize(objetivo, n_trials=13)
    print(f"{nome:10s} melhor x = {estudo.best_params['x']:.3f}   f = {estudo.best_value:.4f}")
print(f"{'verdadeiro':10s} melhor x = {grade[f(grade).argmax()]:.3f}   f = {F_MAX:.4f}")

Uma rodada só é anedota. As mesmas 13 avaliações (3 iniciais + 10), **20 repetições** por amostrador: em
quantas cada um achou o máximo (a menos de 0,001)? Leva um ou dois minutos.

In [ ]:
import time
for nome, fazer in [("aleatório", lambda s: optuna.samplers.RandomSampler(seed=s)),
                    ("TPE", lambda s: optuna.samplers.TPESampler(seed=s, n_startup_trials=3)),
                    ("GP", lambda s: optuna.samplers.GPSampler(seed=s, n_startup_trials=3))]:
    t0, melhores = time.time(), []
    for s in range(20):
        estudo = optuna.create_study(direction="maximize", sampler=fazer(s))
        estudo.optimize(objetivo, n_trials=13)
        melhores.append(estudo.best_value)
    melhores = np.array(melhores)
    print(f"{nome:10s} achou o máximo em {(melhores > F_MAX - 1e-3).sum():2d} de 20 · "
          f"melhor valor médio {br(melhores.mean(), 3)} · {br(time.time() - t0, 1)} s")

## 6. Otimizando hiperparâmetros

O uso mais comum de otimização bayesiana: cada "medida" é **treinar e validar um modelo**.
Aqui, uma floresta aleatória que prevê o limite de escoamento dos aços (Aula 04), com o
erro de validação cruzada como alvo.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

acos = pd.read_csv("data/acos.csv")
Xa, ya = acos.drop(columns="limite_escoamento_MPa"), acos["limite_escoamento_MPa"]

def objetivo_rf(trial):
    modelo = RandomForestRegressor(
        n_estimators=trial.suggest_int("n_estimators", 20, 300, log=True),
        max_depth=trial.suggest_int("max_depth", 2, 20),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
        max_features=trial.suggest_float("max_features", 0.2, 1.0),
        random_state=0, n_jobs=-1)
    return -cross_val_score(modelo, Xa, ya, cv=5, scoring="neg_root_mean_squared_error").mean()

estudo = optuna.create_study(direction="minimize", sampler=optuna.samplers.GPSampler(seed=0))
estudo.optimize(objetivo_rf, n_trials=25)
padrao = -cross_val_score(RandomForestRegressor(random_state=0, n_jobs=-1), Xa, ya, cv=5,
                          scoring="neg_root_mean_squared_error").mean()
print(f"RMSE com os hiperparâmetros padrão: {padrao:.1f} MPa")
print(f"RMSE depois de 25 tentativas:       {estudo.best_value:.1f} MPa  {estudo.best_params}")
rmse = [tr.value for tr in estudo.trials]
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(1, len(rmse) + 1), rmse, "o", color=CINZA, mec="#555555", label="RMSE da tentativa")
ax.step(range(1, len(rmse) + 1), np.minimum.accumulate(rmse), where="post", color=TINTA, lw=2.4,
        label="melhor até agora")
ax.axhline(padrao, color=LARANJA_ESCURO, ls="--", lw=1.6, label="hiperparâmetros padrão")
ax.set_title("Histórico da otimização"); ax.set_xlabel("tentativa"); ax.set_ylabel("RMSE na validação cruzada (MPa)")
virgula(ax); ax.legend();

**O ganho é pequeno — e isso também é resultado.** Florestas aleatórias são pouco
sensíveis aos hiperparâmetros: o padrão do scikit-learn já é bom. A otimização bayesiana
rende mais quando o modelo é sensível (gradient boosting, redes neurais) e quando cada
treino é caro. E cuidado com o número que ela persegue: a seção 7 termina com isso.

## 7. O enigma dos aços — a resposta

> 312 aços. Cada medida custa uma semana. Quantos precisamos medir, em média, para
> achar o mais resistente?

Qual foi o seu palpite na Aula 04? Lá a pergunta era *qual aço ensina mais o modelo*, e escolher
ajudou pouco. Agora o objetivo é outro — **achar o melhor** — e é o problema desta aula.

### Caminho 1: ao acaso, com papel e lápis

Se medimos em ordem aleatória, o melhor aço está em qualquer posição de 1 a N com a mesma
chance. A posição média é `(N + 1) / 2`.

In [ ]:
from sklearn.gaussian_process.kernels import WhiteKernel
from sklearn.preprocessing import StandardScaler

Xe = StandardScaler().fit_transform(acos.drop(columns="limite_escoamento_MPa"))   # composição padronizada
ye = acos["limite_escoamento_MPa"].to_numpy()
N, MELHOR = len(ye), ye.max()
print(f"N = {N} aços · o melhor tem {MELHOR:.1f} MPa (único: {(ye == MELHOR).sum() == 1})")
print(f"ao acaso, analítico: (N + 1)/2 = {(N + 1) / 2:.1f} medidas")

### Caminho 2: ao acaso, simulado

Se as duas rotas não concordarem, há um erro no raciocínio ou no código. É o controle.

In [ ]:
posicoes = [np.flatnonzero(ye[np.random.default_rng(s).permutation(N)] == MELHOR)[0] + 1
            for s in range(2000)]
print(f"ao acaso, 2.000 ordens sorteadas: {np.mean(posicoes):.1f} medidas em média")

As 2.000 ordens servem para conferir a conta. Para comparar com as estratégias de busca, mais adiante, que têm
**20 repetições** cada, o acaso também fica com 20: as ordens das sementes 0 a 19.

In [ ]:
acaso20 = [int(v) for v in posicoes[:20]]                   # as sementes 0 a 19
def media_txt(m):                                           # a média exata de 20 inteiros: 1 casa, ou 2 quando termina em 5
    return br(m, 1) if abs(10 * m - round(10 * m)) < 1e-9 else br(m, 2)
print(f"ao acaso, 20 ordens: {acaso20} · média {media_txt(np.mean(acaso20))} · "
      f"mediana {br(np.median(acaso20), 1)} · de {min(acaso20)} a {max(acaso20)}")

### E se qualquer um dos $k$ melhores servir?

A fórmula geral: com $N$ itens, dos quais $k$ são "bons", a posição média do **primeiro** bom numa
ordem aleatória é

$$\mathbb{E}[\text{posição do primeiro bom}] = \frac{N+1}{k+1}.$$

**Por quê.** Os $k$ bons dividem os $N - k$ ruins em $k + 1$ grupos: os que vêm antes do primeiro bom,
entre o primeiro e o segundo, … , depois do último. Numa ordem aleatória nenhum grupo é privilegiado,
então cada um tem, em média, $(N-k)/(k+1)$ itens. O primeiro bom vem logo depois do primeiro grupo:
$(N-k)/(k+1) + 1 = (N+1)/(k+1)$. Com $k = 1$, é o $(N+1)/2$ acima.

In [ ]:
for k in (1, 5):
    bons = np.argsort(ye)[-k:]                              # os k aços mais resistentes
    sim = np.mean([np.flatnonzero(np.isin(np.random.default_rng(s).permutation(N), bons))[0] + 1
                   for s in range(20_000)])
    print(f"k = {k}: fórmula (N + 1)/(k + 1) = {(N + 1) / (k + 1):6.1f} · 20.000 ordens sorteadas = {sim:6.1f}")

### Com otimização bayesiana

O laço da seção 4, com três mudanças:

* **Um comprimento de correlação por elemento** (Matern 5/2 com *ARD*, *automatic relevance
  determination*: o carbono pode importar numa escala e o cobalto noutra). Com a composição
  $\mathbf{x} = (x_1, \dots, x_{13})$ padronizada, a distância que entra no Matern da seção 1 passa a ter
  um $\ell_d$ por elemento:

  $$r(\mathbf{x}, \mathbf{x}') = \sqrt{\sum_{d=1}^{13} \frac{(x_d - x'_d)^2}{\ell_d^2}} .$$

  Um $\ell_d$ pequeno quer dizer que a resistência muda depressa com o elemento $d$; um $\ell_d$ enorme,
  que ele quase não importa. No código, `Matern(length_scale=np.ones(13))`: um vetor, não um número.
* **Um termo de ruído** (`WhiteKernel`): soma $\sigma_n^2$ à diagonal — o `alpha` da seção 2, agora
  estimado, porque aços de composição quase igual têm resistências diferentes. E `normalize_y=True` é
  seguro aqui: com dezenas de medidas, a média e o desvio delas são estimativas razoáveis.
* **Um conjunto finito de candidatos.** Na seção 4 o candidato era qualquer $x$ de uma grade; aqui é um
  dos aços da tabela **ainda não medidos**. O EI é calculado em todos eles e escolhe-se o maior.

Começamos com 10 aços ao acaso (que contam no total) e paramos quando o melhor aparece. Uma
repetição só seria anedota: rodamos **20 repetições** (sementes), cada uma com outros 10 iniciais.
Leva de um a quatro minutos, conforme a máquina.

In [ ]:
import time

def bo_acos(semente, n0=10, orcamento=150):
    rng = np.random.default_rng(semente)
    medidos = list(rng.choice(N, n0, replace=False))
    historico = [ye[medidos].max()]
    while ye[medidos].max() < MELHOR and len(medidos) < orcamento:
        kernel = (ConstantKernel(1.0) * Matern(length_scale=np.ones(Xe.shape[1]), nu=2.5)
                  + WhiteKernel(1e-2))
        gp = GaussianProcessRegressor(kernel, normalize_y=True, random_state=0)
        gp.fit(Xe[medidos], ye[medidos])
        resto = np.setdiff1d(np.arange(N), medidos)
        mu, sd = gp.predict(Xe[resto], return_std=True)
        f_star = ye[medidos].max()
        z = (mu - f_star) / np.maximum(sd, 1e-9)
        ei = (mu - f_star) * norm.cdf(z) + sd * norm.pdf(z)
        medidos.append(resto[np.argmax(ei)])
        historico.append(ye[medidos].max())
    return len(medidos), historico

t0 = time.time()
rodadas = [bo_acos(s) for s in range(20)]
n_bo = np.array([r[0] for r in rodadas])
print("medidas até achar o melhor, por repetição:", [int(v) for v in n_bo])
print(f"média {n_bo.mean():.1f} · mediana {np.median(n_bo):.1f} · de {n_bo.min()} a {n_bo.max()}")
print(f"menos medidas que a média ao acaso ({(N + 1) / 2:.1f}) em {(n_bo < (N + 1) / 2).sum()} de 20 repetições"
      f" · pararam no orçamento de 150 sem achar: {(n_bo >= 150).sum()}   ({time.time() - t0:.0f} s)")

### Comparando com o modelo da Aula 04

Na Aula 04, nos mesmos aços, o modelo era uma **floresta aleatória** de 50 árvores, e a escolha era o
**desacordo das árvores** (o desvio entre elas): medir onde o modelo sabe pouco — para aprender o modelo
inteiro. Aqui o objetivo é outro: achar o melhor. Comparamos, nas **mesmas 20 repetições** (os mesmos
10 aços iniciais em cada uma), três regras com a floresta:

* **desacordo das árvores** — a da Aula 04: só explora;
* **o maior previsto** (a média das árvores) — gulosa: só explota;
* **UCB com a floresta** — a média das árvores + 2 × o desvio delas: a regra de hoje, com o modelo da Aula 04.

Leva cerca de um minuto.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

def floresta_acos(semente, regra, n0=10, kappa=2.0):
    rng = np.random.default_rng(semente)
    medidos = list(rng.choice(N, n0, replace=False))          # os mesmos 10 iniciais do GP + EI
    while ye[medidos].max() < MELHOR:
        modelo = RandomForestRegressor(n_estimators=50, random_state=0, n_jobs=1).fit(Xe[medidos], ye[medidos])
        resto = np.setdiff1d(np.arange(N), medidos)
        arvores = np.stack([a.predict(Xe[resto]) for a in modelo.estimators_])
        media, desvio = arvores.mean(axis=0), arvores.std(axis=0)
        nota = {"desacordo": desvio, "gulosa": media, "ucb": media + kappa * desvio}[regra]
        medidos.append(resto[np.argmax(nota)])
    return len(medidos)

t0 = time.time()
n_por_regra = {"floresta: desacordo das árvores": [floresta_acos(s, "desacordo") for s in range(20)],
               "floresta: o maior previsto": [floresta_acos(s, "gulosa") for s in range(20)],
               "floresta: UCB": [floresta_acos(s, "ucb") for s in range(20)],
               "GP + EI": [int(v) for v in n_bo]}
for nome, ns in n_por_regra.items():
    print(f"{nome}: {ns} · média {np.mean(ns):.1f} · mediana {np.median(ns):.1f} · de {min(ns)} a {max(ns)}")
for nome, ns in n_por_regra.items():
    if nome != "GP + EI":
        menos = sum(a < b for a, b in zip(n_por_regra["GP + EI"], ns))
        empate = sum(a == b for a, b in zip(n_por_regra["GP + EI"], ns))
        print(f"GP + EI × {nome}: precisou de menos medidas em {menos} de 20 repetições (empates: {empate})")
print(f"({time.time() - t0:.0f} s)")

In [ ]:
fig, (a0, a1) = plt.subplots(1, 2, figsize=(15, 5.2), gridspec_kw=dict(width_ratios=[1, 1.15]))
for _, h in rodadas:                                   # cada linha: uma repetição do GP + EI
    a0.plot(np.arange(10, 10 + len(h)), h, color=TINTA, alpha=0.45, lw=1.4)
media_acaso = np.mean([np.maximum.accumulate(ye[np.random.default_rng(s).permutation(N)])
                       for s in range(500)], axis=0)
a0.plot(np.arange(1, N + 1), media_acaso, color="#7A7A7A", ls="--", lw=2.4)
a0.axhline(MELHOR, color="k", ls=":", lw=1.5)
a0.text(158, media_acaso[99] - 45, "ao acaso\n(média de 500 ordens)", ha="right", va="top", color="#555555",
        fontweight="bold")
a0.text(70, 1850, "linhas escuras: GP + EI,\n20 repetições, uma linha cada", va="top", color=TINTA, fontweight="bold")
a0.text(158, MELHOR + 12, "o melhor aço", ha="right", va="bottom")
a0.set_xlim(0, 160); a0.set_ylim(top=MELHOR + 90)
a0.set_xlabel("aços medidos"); a0.set_ylabel("melhor até agora (MPa)"); virgula(a0)

linhas = [("ao acaso", acaso20, "o", CINZA),
          ("floresta: desacordo\ndas árvores (Aula 04)", n_por_regra["floresta: desacordo das árvores"], "s", "#7A7A7A"),
          ("floresta: o maior\nprevisto", n_por_regra["floresta: o maior previsto"], "^", "#7A7A7A"),
          ("floresta: UCB", n_por_regra["floresta: UCB"], "D", TINTA),
          ("GP + EI", n_por_regra["GP + EI"], "o", TINTA)]
g = np.random.default_rng(1)
for y0, (nome, ns, marca, cor) in enumerate(linhas):         # 20 repetições em cada fileira
    ns = np.asarray(ns)
    a1.scatter(ns, y0 + g.uniform(-0.15, 0.15, len(ns)), s=55, marker=marca, color=cor, edgecolor="white", zorder=3)
    m = ns.mean()
    a1.plot([m, m], [y0 - 0.38, y0 + 0.38], color="k", lw=2.5, zorder=4)
    a1.text(m + 5, y0 + 0.22, f"média {media_txt(m)}", fontweight="bold", va="bottom", fontsize=11)
a1.set_yticks(range(len(linhas)), [nome for nome, *_ in linhas])
a1.set_ylim(-0.6, len(linhas) - 0.3); a1.set_xlim(0, N + 8)
a1.set_title("cada fileira: 20 repetições", loc="left", fontsize=12, color=TINTA)
a1.set_xlabel("medidas até achar o melhor aço"); a1.xaxis.set_major_formatter(VIRGULA)
plt.tight_layout()

**A resposta.** Ao acaso, as duas rotas dão ~156 medidas. Com otimização bayesiana, compare a média
das 20 repetições acima — as 10 iniciais já estão incluídas. A diferença entre as repetições é grande
(veja os pontos da direita): uma repetição só não diria quanto se economiza. Cada fileira tem 20 repetições,
inclusive a do acaso — e com só 20 ordens o acaso às vezes tem sorte (os pontos mais à esquerda da fileira dele).

**E o modelo da Aula 04?** Três coisas na figura: (1) toda regra que usa o modelo ganha de longe do
acaso; (2) com uma regra que equilibra explorar e explotar (UCB, EI), a floresta e o GP ficam parecidos
— o que mais pesa é a regra, não o modelo; (3) a gulosa tem as repetições mais longas: sem explorar,
às vezes fica presa perto de um aço bom que não é o melhor. O desacordo das árvores, que só explora,
também acha relativamente depressa aqui: o aço mais resistente está numa região de composições rara,
justamente onde as árvores discordam.

O que o GP aprendeu sobre os elementos? Ajustado aos 312 aços, os $\ell_d$ dizem quais elementos
ele achou relevantes:

In [ ]:
kernel_ard = ConstantKernel(1.0) * Matern(length_scale=np.ones(Xe.shape[1]), nu=2.5) + WhiteKernel(1e-2)
gp_todos = GaussianProcessRegressor(kernel_ard, normalize_y=True, random_state=0).fit(Xe, ye)
ells = gp_todos.kernel_.k1.k2.length_scale
elementos = acos.drop(columns="limite_escoamento_MPa").columns
print("comprimento de correlação por elemento (composição padronizada) — menor = mais relevante:")
for el, l in sorted(zip(elementos, ells), key=lambda par: par[1]):
    print(f"  {el:3s} ℓ = {l:9.2f}")

dez = acos.nlargest(10, "limite_escoamento_MPa")
demais = acos.drop(dez.index)
print()
print("composição média (% em massa): os 10 mais resistentes × os demais")
print(pd.DataFrame({"10 mais resistentes": dez[["ni", "co", "mo", "ti", "cr"]].mean(),
                    "demais": demais[["ni", "co", "mo", "ti", "cr"]].mean()}).round(2))

Os elementos de $\ell$ menor são os que o modelo usa para distinguir aços fortes de fracos — e a
tabela de composição confirma: os dez aços mais resistentes têm muito mais níquel, cobalto,
molibdênio e titânio que os demais, e quase nenhum cromo. É a receita dos aços *maraging*. Dois
cuidados: **relevância não é causa** (o modelo só vê correlação na tabela), e o $\ell$ depende da
padronização — um elemento que quase não varia na tabela não tem como parecer relevante.

Três perguntas para fechar o enigma:
1. Nas curvas da esquerda, a otimização bayesiana às vezes fica parada num patamar. O que
   ela está fazendo nessas medidas?
2. Por que o EI escolhe o próximo **entre os aços existentes**, e não uma composição
   qualquer? O que mudaria se pudéssemos fabricar composições novas?
3. O que aconteceria se as medidas tivessem ruído de ±100 MPa?

**A lição do bloco.** A otimização bayesiana é muito boa em otimizar o número que você dá a ela.
Escolher esse número — qual métrica, medida em quantas janelas, com quantas repetições — é o seu
trabalho, e é onde mora quase todo erro. Um exemplo: ao escolher os *lags* da regressão das séries
do M4 (Aula 03) olhando uma janela de validação só, a otimização melhora a validação e piora o teste — ela
decora aquelas 48 horas. É o sobreajuste da Aula 02, um nível acima. No enigma dos aços isso não
acontece porque o número perseguido é a própria resistência medida.

## 8. Limites, medidos

Quatro limites da otimização bayesiana com GP. Os três primeiros, medidos.

### Muitas variáveis: tudo fica longe de tudo

**Variáveis** são as entradas do GP — o que se escolhe ao decidir onde medir: na função desta aula, 1 (o x);
no enigma, 13 (os elementos da composição); na floresta da seção 6, 4 hiperparâmetros.
O GP decide pela distância: perto de uma medida ele sabe, longe não sabe. Com 50 medidas sorteadas
num cubo de $d$ variáveis (cada uma entre 0 e 1), qual a distância de um ponto qualquer até a medida
**mais próxima**, comparada com a distância entre dois pontos quaisquer?

In [ ]:
g = np.random.default_rng(0)
dims = [1, 2, 5, 10, 20, 50]
perto, qualquer = [], []
for d in dims:
    medidas, pontos = g.uniform(size=(50, d)), g.uniform(size=(1000, d))
    dist = np.sqrt(((pontos[:, None, :] - medidas[None]) ** 2).sum(-1))
    perto.append(np.median(dist.min(axis=1))); qualquer.append(np.median(dist))
    print(f"d = {d:2d}: até a medida mais próxima {br(perto[-1])} · entre dois pontos quaisquer {br(qualquer[-1])}"
          f" · razão {br(perto[-1] / qualquer[-1])}")
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(dims, np.array(perto) / np.array(qualquer), "o-", color=TINTA, lw=2.4, ms=8)
ax.axhline(1, color="k", ls=":", lw=1.2)
ax.text(dims[-1], 1.03, "tão longe quanto um ponto qualquer", ha="right", va="bottom")
ax.set_xscale("log"); ax.set_xticks(dims, [str(d) for d in dims])
ax.set_ylim(0, 1.15); ax.yaxis.set_major_formatter(VIRGULA)
ax.set_xlabel("número de variáveis d"); ax.set_ylabel("distância até a medida mais próxima\n÷ distância típica")
ax.set_title("50 medidas: com muitas variáveis, nenhum ponto está perto de uma medida");

Com uma variável, a medida mais próxima está a 1/50 do caminho; com 20 ou 50, está quase tão longe
quanto um ponto qualquer. A faixa do GP fica larga em todo lugar, e a aquisição perde o que escolher.
Quantas variáveis dá para tratar **depende do orçamento**: mais variáveis pedem muito mais medidas.
A regra prática da literatura é de menos de ~20 variáveis (Frazier, 2018). Saídas: escolher as
variáveis que importam (o $\ell$ por variável da seção 7 mostra quais), ou modelos e regiões de
confiança feitos para alta dimensão (TuRBO).

### Muitas medidas: o custo explode

Para calcular μ e σ, o GP resolve um sistema de **n equações com n incógnitas** (n = número de medidas), com a
matriz $n \times n$ das covariâncias entre todas as medidas. A eliminação (a fatoração de Cholesky) zera uma coluna
por vez, e cada uma mexe num bloco de até $n \times n$ números: cerca de $n^3/3$ operações. Dobrar $n$ → 8 vezes mais.
Medimos o tempo de um ajuste (kernel fixo, sem otimizar $\ell$, **um núcleo só**, para a medida não depender
de como a biblioteca divide o trabalho entre os núcleos) para $n$ crescente:

In [ ]:
from threadpoolctl import threadpool_limits      # um núcleo só: com vários, o tempo depende da divisão do trabalho

tempos = []
ns = [250, 500, 1000, 2000, 4000]
g = np.random.default_rng(0)
for n in ns:
    Xn = g.uniform(size=(n, 2)); yn = np.sin(6 * Xn).sum(1)
    melhor = np.inf
    for _ in range(3):
        with threadpool_limits(limits=1):
            t0 = time.perf_counter()
            GaussianProcessRegressor(Matern(length_scale=0.3, nu=2.5), alpha=1e-6, optimizer=None).fit(Xn, yn)
            melhor = min(melhor, time.perf_counter() - t0)
    tempos.append(melhor)
inclinacao = np.log(tempos[-1] / tempos[-2]) / np.log(2)
print(f"tempos (s): " + " · ".join(f"n = {n}: {br(t, 3)}" for n, t in zip(ns, tempos)))
print(f"de 2.000 para 4.000 medidas: tempo × {br(tempos[-1] / tempos[-2], 1)} (inclinação {br(inclinacao, 1)}; n³ daria × 8)")
fig, ax = plt.subplots(figsize=(9, 4))
ax.loglog(ns, tempos, "o-", color=TINTA, lw=2.4, ms=8)
ref = tempos[-1] * (np.array(ns) / ns[-1]) ** 3
ax.loglog(ns, ref, "--", color="#7A7A7A", lw=1.6)
ax.text(ns[1], ref[1], "  cresce com n³", color="#555555", va="top")
ax.set_xticks(ns, [str(n) for n in ns]); ax.minorticks_off()
ax.set_xlabel("número de medidas n"); ax.set_ylabel("tempo de um ajuste (s)")
ax.set_title("dobrar as medidas multiplica o tempo");

Ótimo para 100 experimentos caros; ruim para 100 mil avaliações baratas — aí outros métodos ganham
(aproximações esparsas do GP, ou outros modelos, como as florestas da seção 7).

### Medidas com ruído

Medidas reais têm ruído. A mesma função da aula, medida em 12 pontos com ruído de desvio 0,3, ajustada
**sem** e **com** um termo de ruído no kernel (`WhiteKernel`, que soma $\sigma_n^2$ à diagonal e é ajustado aos dados):

In [ ]:
g = np.random.default_rng(5)
Xr = np.sort(g.uniform(A, B, 12)); yr = f(Xr) + g.normal(0, 0.3, 12)
sem = ajustar_gp(Xr, yr)
k_ruido = (ConstantKernel(1.0, "fixed") * Matern(length_scale=1.0, length_scale_bounds=(0.2, 2.0), nu=2.5)
           + WhiteKernel(0.05, noise_level_bounds=(1e-4, 1.0)))
com = GaussianProcessRegressor(k_ruido, n_restarts_optimizer=5, random_state=0).fit(Xr[:, None], yr)
fig, eixos = plt.subplots(1, 2, figsize=(15, 4), sharey=True)
for ax, gp_r, titulo in [(eixos[0], sem, "sem termo de ruído: acredita em cada medida"),
                         (eixos[1], com, "com termo de ruído: a faixa não se fecha")]:
    m_r, s_r = gp_r.predict(grade[:, None], return_std=True)
    ax.plot(grade, f(grade), "k--", lw=1)
    ax.fill_between(grade, m_r - 2 * s_r, m_r + 2 * s_r, color=AZUL, alpha=0.18, lw=0)
    ax.plot(grade, m_r, color=TINTA, lw=2)
    ax.plot(Xr, yr, "o", color=LARANJA, mec="#1A1A1A", ms=9, zorder=3)
    ax.set_title(titulo); ax.set_xlabel("x"); virgula(ax)
eixos[0].set_ylabel("f(x)"); eixos[0].set_ylim(-3, 3)           # sem o termo, a média chega a mergulhar fora da figura
melhor_medido = Xr[yr.argmax()]
print(f"ruído estimado pelo GP: desvio {br(np.sqrt(com.kernel_.k2.noise_level))} (o verdadeiro é 0,3)")
print(f"o 'melhor medido' está em x = {br(melhor_medido)}, onde a função vale {br(f(melhor_medido))}; "
      f"o maior valor da função entre os 12 pontos é {br(f(Xr).max())}")

Sem o termo de ruído, a média serpenteia para passar por cada medida e a faixa se fecha sobre medidas
que não são exatas. Com ele, o GP estima o ruído (perto dos 0,3 verdadeiros) e a faixa não se fecha.
Outro cuidado: com ruído, o recorde $f^*$ — o melhor valor **medido** — pode ser sorte de uma medida alta
(aqui não foi: veja a saída acima); por isso se usa o melhor valor **previsto** entre os medidos.

### Variáveis discretas e restrições

* **Inteiras** — o número de árvores, a profundidade e o mínimo de amostras por folha da floresta da seção 6
  (`suggest_int`), o número de *lags* da Aula 03: têm ordem e distância; o `GPSampler` do Optuna as trata
  arredondando para o inteiro mais próximo.
* **Categóricas** — o catalisador A, B ou C; usar ou não o calendário como entrada (Aula 03); kernel RBF ou
  Matern (`suggest_categorical`): não têm ordem. Para o kernel do `GPSampler`, a distância é só "igual" (0) ou
  "diferente" (1) — uma noção pobre de perto. O TPE trata as duas naturalmente.
* **Restrições:** "a maior resistência **com** alongamento acima de 10%". O **alongamento** é o quanto o aço
  estica antes de romper no ensaio de tração, em % do comprimento: (L_final − L₀)/L₀ × 100. Abaixo de ~10%, o aço
  é frágil — quebra sem avisar. É preciso um segundo modelo (a chance de cumprir a restrição) multiplicado na
  aquisição. Nos nossos aços a restrição muda a resposta do enigma:

In [ ]:
import gzip, json
bruto = json.load(gzip.open("data/steel_strength.json.gz"))      # a tabela original, com o alongamento
tabela = pd.DataFrame(bruto["data"], columns=bruto["columns"])
along, escoa = tabela["elongation"], tabela["yield strength"]
mais = tabela.loc[escoa.idxmax()]
ok = tabela[along > 10]
print(f"alongamento: de {br(along.min(), 1)} a {br(along.max(), 1)}%, mediana {br(along.median(), 1)}% "
      f"(falta em {along.isna().sum()} dos {len(tabela)}) · correlação resistência × alongamento {br(tabela[['yield strength', 'elongation']].corr().iloc[0, 1])}")
print(f"o mais resistente: {br(mais['yield strength'], 1)} MPa, alongamento {br(mais['elongation'], 1)}% — não cumpre a restrição")
print(f"com alongamento acima de 10%: {len(ok)} de {along.notna().sum()}; o mais resistente deles tem {br(ok['yield strength'].max(), 1)} MPa")

## 9. Para ir além

* **Ruído.** Com medidas ruidosas, $\sigma_n^2$ deixa de ser um número minúsculo e passa a ser estimado
  (`+ WhiteKernel()`). A faixa não se fecha mais sobre as medidas, e o "melhor valor medido" $f^*$ deixa
  de ser confiável: usa-se o melhor valor **previsto** entre os medidos.
* **Várias dimensões.** Com $d$ variáveis, um $\ell$ por variável (*ARD*) deixa o GP descobrir quais
  importam — ele aparece na seção 7, nos 13 elementos dos aços.
* **Otimização em lote.** No laboratório se mede de 8 em 8. Escolher 8 pontos de uma vez exige que a
  aquisição "imagine" os resultados dos primeiros para escolher os seguintes (q-EI, *kriging believer*).
* **Visualização interativa:** Görtler e colegas, "A Visual Exploration of Gaussian Processes",
  *Distill*, 2019 — https://distill.pub/2019/visual-exploration-gaussian-processes

### Referências
* Rasmussen, C. E.; Williams, C. K. I. *Gaussian Processes for Machine Learning*. MIT Press, 2006.
  Gratuito em https://gaussianprocess.org/gpml/ — a referência para as seções 1 e 2.
* Frazier, P. I. A tutorial on Bayesian optimization. arXiv:1807.02811, 2018 — curto e claro.
* Shahriari, B. e colegas. Taking the human out of the loop: a review of Bayesian optimization.
  *Proceedings of the IEEE*, 104(1), 2016.
* Kushner, H. J. A new method of locating the maximum point of an arbitrary multipeak curve in the
  presence of noise. *Journal of Basic Engineering*, 86(1), 1964 — a origem da PI.
* Jones, D. R.; Schonlau, M.; Welch, W. J. Efficient global optimization of expensive black-box
  functions. *Journal of Global Optimization*, 13(4), 1998 — o EI como se usa hoje.
* Srinivas, N. e colegas. Gaussian process optimization in the bandit setting: no regret and
  experimental design. *ICML*, 2010 — o GP-UCB.
* Bergstra, J. e colegas. Algorithms for hyper-parameter optimization. *NeurIPS*, 2011 — o TPE.

---
# Exercício: uma função caixa-preta

Uma função **caixa-preta** de duas variáveis com **orçamento de 20 avaliações**. Cada avaliação é um
"experimento": a 21ª dá erro.

```python
from caixa_preta import CaixaPreta
f = CaixaPreta()
valor = f(0.3, 0.7)        # x e y em [0, 1]
f.restantes, f.melhor, f.historico
```

**Objetivo:** o maior valor possível dentro do orçamento (`f.melhor`).

1. Antes de gastar a primeira avaliação, **escreva a estratégia**: quantos pontos
   iniciais, como espalhados, qual aquisição, qual `κ` ou `ξ`.
2. Rode (à mão, com o seu laço, ou com o Optuna usando a sua `CaixaPreta` dentro do
   objetivo).
3. Plote os 20 pontos avaliados sobre o plano `x × y`, numerados na ordem.
4. Responda: a sua estratégia explorou ou explotou mais? Houve um momento em que ela
   "decidiu" onde estava o máximo? Foi a decisão certa?

**Cuidados que fazem diferença:**
* Escreva a estratégia **antes** de gastar a primeira avaliação.
* Criar outra `CaixaPreta` é começar de novo com mais 20 avaliações — vale para comparar estratégias
  (várias repetições!), não para "achar mais".
* Sorte conta: explorar bem e dar azar acontece. O que interessa é entender, pelo gráfico dos 20
  pontos, quando a sua estratégia explorou e quando explotou.

---
# Apresentações de 19/11

Em **duplas**, **10 minutos** por dupla. Escolham **um** problema do bloco — de **séries temporais** (Aulas 01–03) **ou** de
**aprendizado ativo / otimização bayesiana** (Aulas 04–05) — e apresentem o que fizeram. Não há
nada a entregar: só a apresentação.

O que mostrar:
1. **O problema**: os dados e a pergunta.
2. **O que vocês fizeram** — e a referência com que compararam (a ingênua sazonal, a escolha
   aleatória…).
3. **O resultado medido**, com várias repetições quando houver sorte no meio ("ganhou em 15 de 20").
4. **Uma armadilha** que encontraram, ou um caso em que o método não ajudou.

Pontos de partida no material: a previsão das séries do M4 (Aula 03), o aprendizado ativo da Aula 04, a
função caixa-preta (acima), o enigma dos aços com outro objetivo (por exemplo, com a restrição de
alongamento), ou um problema da IC/TCC de vocês.